In [1]:
from blazingsql import BlazingContext
from matplotlib import pyplot as plt
import cudf
import time
import numpy

bc_cudf = BlazingContext()

BlazingContext ready


In [2]:
%%time
# Load CSVs into GPU DataFrames with cudf (GDF)
netflow_gdf_cudf = cudf.read_csv('./data/final.csv')
"""cp1_netflow = cudf.concat([netflow_gdf_cudf, netflow_gdf_cudf], ignore_index=True)
del netflow_gdf_cudf
cp2_netflow = cudf.concat([cp1_netflow, cp1_netflow], ignore_index=True)
del cp1_netflow"""

CPU times: user 12.2 s, sys: 7.33 s, total: 19.5 s
Wall time: 34.9 s


'cp1_netflow = cudf.concat([netflow_gdf_cudf, netflow_gdf_cudf], ignore_index=True)\ndel netflow_gdf_cudf\ncp2_netflow = cudf.concat([cp1_netflow, cp1_netflow], ignore_index=True)\ndel cp1_netflow'

In [3]:
%%time
# Create BlazingSQL table from GDF
bc_cudf.create_table('netflow', netflow_gdf_cudf)
# bc_cudf.create_table('netflow_B', cp1_netflow)

# del netflow_gdf_cudf
# del cp1_netflow

CPU times: user 7.27 ms, sys: 1.94 ms, total: 9.2 ms
Wall time: 8.29 ms


In [5]:
df_cudf = cudf.DataFrame(netflow_gdf_cudf)

NameError: name 'netflow_gdf_cudf' is not defined

In [4]:
netflow_gdf_cudf.head()

,TimeSeconds,parsedDate,dateTimeStr,ipLayerProtocol,ipLayerProtocolCode,firstSeenSrcIp,firstSeenDestIp,firstSeenSrcPort,firstSeenDestPort,moreFragments,contFragments,durationSeconds,firstSeenSrcPayloadBytes,firstSeenDestPayloadBytes,firstSeenSrcTotalBytes,firstSeenDestTotalBytes,firstSeenSrcPacketCount,firstSeenDestPacketCount,recordForceOut
0,1.364948e+09,2013-04-03 00:11:50,2.013040e+13,6,TCP,10.38.37.13,172.20.0.3,42559,25,0,0,10,36,125,422,403,7,5,0
1,1.364948e+09,2013-04-03 00:11:53,2.013040e+13,6,TCP,10.13.77.49,172.30.0.4,42566,25,0,0,9,0,0,186,0,3,0,0
2,1.364948e+09,2013-04-03 00:11:54,2.013040e+13,17,UDP,172.10.0.40,172.255.255.255,138,138,0,0,0,201,0,243,0,1,0,0
3,1.364948e+09,2013-04-03 00:11:57,2.013040e+13,6,TCP,10.156.215.83,172.10.0.7,42593,80,0,0,0,170,336,448,506,5,3,0
4,1.364948e+09,2013-04-03 00:12:00,2.013040e+13,6,TCP,10.170.32.110,172.20.0.4,42612,80,0,0,3,1870,79850,5730,84250,70,80,0


In [ ]:
%%time
num_runs = 10

# make a query
queries = []
queries.append('''SELECT a.firstSeenSrcIP as source FROM netflow a''')
queries.append('''SELECT count(a.firstSeenDestPort) as targetPorts FROM netflow a''')
queries.append('''SELECT SUM(a.firstSeenSrcTotalBytes) as bytesOut FROM netflow a''')
# queries.append('''SELECT AVG(a.firstSeenDestTotalBytes) as bytesIn FROM netflow a''')
# queries.append('''SELECT DISTINCT(a.durationSeconds) as durationSeconds FROM netflow a''')
queries.append('''SELECT MIN(parsedDate) as firstFlowDate FROM netflow a''')
queries.append('''SELECT MAX(parsedDate) as lastFlowDate FROM netflow a''')
queries.append('''SELECT COUNT(*) as attemptCount FROM netflow a''')
queries.append('''SELECT a.firstSeenSrcTotalBytes, a.firstSeenDestTotalBytes FROM netflow a ORDER BY a.firstSeenSrcTotalBytes''')
# queries.append('''SELECT netflow a.firstSeenSrcPacketCount, netflow_B a.firstSeenSrcPacketCount, netflow_B a.firstSeenDestPacketCount 
#                FROM netflow a 
#                INNER JOIN netflow_B a
#                ON (netflow a.firstSeenSrcPacketCount = netflow_B a.recordForceOut)''')

all_times=[]
for query in queries:
    times=[]
    result0 = ""
    print("\nquery:", query)
    for i in range(0,num_runs):
        print("start")
        t0 = time.time()
        result = bc_cudf.sql(query)
        t2 = time.time()
        times.append(t2 - t0)
        # result0 = result
        del result
        time.sleep(10)
        print(i+1, "query done")
    # print(result0)
    all_times.append(times)
    for j in range(0,num_runs):
        print("times[%d]:" %j + str(times[j]))


query: SELECT a.firstSeenSrcIP as source FROM netflow a
start
1 query done
start
2 query done
start
3 query done
start
4 query done
start
5 query done
start
6 query done
start
7 query done
start
8 query done
start
9 query done
start
10 query done
times[0]:5.029048919677734
times[1]:0.07903695106506348
times[2]:0.04511141777038574
times[3]:0.0477447509765625
times[4]:0.0456085205078125
times[5]:0.04555773735046387
times[6]:0.044548988342285156
times[7]:0.04510664939880371
times[8]:0.0449066162109375
times[9]:0.045156002044677734

query: SELECT count(a.firstSeenDestPort) as targetPorts FROM netflow a
start
1 query done
start
2 query done
start
3 query done
start
4 query done
start
5 query done
start
6 query done
start
7 query done
start
8 query done
start
9 query done
start
10 query done
times[0]:0.034639596939086914
times[1]:0.019640684127807617
times[2]:0.019922971725463867
times[3]:0.018096208572387695
times[4]:0.019201278686523438
times[5]:0.026013612747192383
times[6]:0.01974225044

In [4]:
num_runs = 10
query = '''
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            AVG(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            '''

all_times=[]
times=[]
result0 = ""
print("\nquery:", query)
for i in range(0,num_runs):
   t0 = time.time()
   result = bc_cudf.sql(query)
   t2 = time.time()
   times.append(t2 - t0)
   result0 = result
   del result
   time.sleep(1)
print(result0)
all_times.append(times)
for j in range(0,num_runs):
   print("times[%d]:" %j + str(times[j]))


query: 
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            AVG(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            
             source      destination  targetPorts  bytesOut       bytesIn  \
0       172.30.1.70         10.0.0.9           79     34894  6.059494e+02   
1        172.10.1.1        10.0.0.14          102     46416  6.324706e+02   
2        172.30.1.1         10.0.0.5           69     31195  6.295362e+02   
3      172.10.1.234         10.0.0.5          104     47287  6.225962e+02   
4       172.10.1.81  239.255.255

In [5]:
sums = []
# for (times, query) in (all_times, queries):
for times in all_times:
    sum = 0.0
    for time_result in (times):
        sum += time_result
    print("query: " + query)
    print("총합: " + str(sum))
    print("평균: " + str(numpy.mean(times)))
    print("분산: " + str(numpy.var(times)))
    print("표준편차: " + str(numpy.std(times)))
    print("\n")

query: 
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            AVG(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            
총합: 2.893617630004883
평균: 0.28936176300048827
분산: 0.3759834898897202
표준편차: 0.6131749260119173




In [ ]:
gdf

In [11]:
# how's it look?
df_times = cudf.DataFrame(all_times)
df_times.to_csv("time_output.csv", index=False)